# 2012 Round 2 Asset Schedule Analysis

This notebook builds the forecast asset schedules and analyzes the impact on deferred tax.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ============================================================
# DATA SETUP
# ============================================================

# Read the introduction to extract parameters
import os

data_dir = '/workspace/data'
if not os.path.exists(data_dir):
    # Fallback for local testing
    data_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'environment', 'data')

intro_path = os.path.join(data_dir, 'introduction.txt')
if os.path.exists(intro_path):
    with open(intro_path, 'r') as f:
        intro_text = f.read()
    print(intro_text)
else:
    print("Introduction file not found, using hardcoded data from the problem statement.")

# Real 2012 US$m capex values from the introduction
years = list(range(2012, 2030))
real_capex_values = [10, 64, 69, 99, 89, 39, 34, 23, 29, 69, 99, 89, 39, 34, 23, 29, 69, 99]

# Create a DataFrame with capex data
capex_df = pd.DataFrame({
    'Year': years,
    'Real_Capex': real_capex_values
})

# Parameters from the introduction
inflation_rate = 0.03  # 3% per annum
acct_dep_life = 12     # Straight-line over 12 years
tax_dep_rate = 0.40    # Diminishing value at 40% p.a.
# Tax rate: 30% until end of 2015, 28% from start of 2016

print(f"Inflation rate: {inflation_rate*100}%")
print(f"Accounting depreciation life: {acct_dep_life} years")
print(f"Tax depreciation rate: {tax_dep_rate*100}%")
print(f"Tax rate: 30% (2012-2015), 28% (2016 onwards)")
print()
print(capex_df.to_string(index=False))

In [ ]:
# ============================================================
# NOMINAL CAPEX CALCULATION
# ============================================================
# Capex is in real 2012 dollars. To convert to nominal:
# Nominal = Real * (1 + inflation)^(year - 2012)
# Capex is incurred on the first day of each calendar year

capex_df['Inflation_Factor'] = (1 + inflation_rate) ** (capex_df['Year'] - 2012)
capex_df['Nominal_Capex'] = capex_df['Real_Capex'] * capex_df['Inflation_Factor']

print("Nominal Capex Schedule:")
print(capex_df[['Year', 'Real_Capex', 'Inflation_Factor', 'Nominal_Capex']].to_string(index=False))

# Build a dictionary for easy lookup
nominal_capex = dict(zip(capex_df['Year'], capex_df['Nominal_Capex']))

In [ ]:
# ============================================================
# QUESTION 1: Total nominal capex for 2012 through 2021 (inclusive)
# ============================================================
# Options: a) 593.0m  b) 598.3m  c) 525.0m  d) 583.3m

total_nominal_2012_2021 = sum(nominal_capex[y] for y in range(2012, 2022))
print(f"Total nominal capex 2012-2021: {total_nominal_2012_2021:.1f}m")

# Compare with options
options_q1 = {'a': 593.0, 'b': 598.3, 'c': 525.0, 'd': 583.3}
q1_answer = min(options_q1, key=lambda k: abs(options_q1[k] - total_nominal_2012_2021))
print(f"Closest match: {q1_answer}) {options_q1[q1_answer]}m")

In [ ]:
# ============================================================
# ACCOUNTING ASSET SCHEDULE
# ============================================================
# Straight-line depreciation over 12 years on nominal capex
# Capex incurred on first day of year, so first year gets full depreciation

def acct_closing_balance(target_year):
    """Calculate closing accounting asset balance at end of target_year."""
    total = 0.0
    for vy in years:
        if vy > target_year:
            continue
        nc = nominal_capex[vy]
        dep_per_year = nc / acct_dep_life
        years_depr = target_year - vy + 1  # full year in first year
        if years_depr > acct_dep_life:
            years_depr = acct_dep_life
        remaining = nc - dep_per_year * years_depr
        if remaining < 0:
            remaining = 0
        total += remaining
    return total

print("Accounting Asset Schedule - Closing Balances:")
acct_balances = {}
for y in years:
    bal = acct_closing_balance(y)
    acct_balances[y] = bal
    print(f"  {y}: {bal:.1f}m")

In [ ]:
# ============================================================
# QUESTION 2: Closing balance for accounting asset schedule in 2017
# ============================================================
# Options: a) 298.3m  b) 286.7m  c) 370.0m  d) 331.0

acct_close_2017 = acct_closing_balance(2017)
print(f"Accounting closing balance 2017: {acct_close_2017:.1f}m")

options_q2 = {'a': 298.3, 'b': 286.7, 'c': 370.0, 'd': 331.0}
q2_answer = min(options_q2, key=lambda k: abs(options_q2[k] - acct_close_2017))
print(f"Closest match: {q2_answer}) {options_q2[q2_answer]}m")

In [ ]:
# ============================================================
# TAXATION ASSET SCHEDULE
# ============================================================
# Diminishing value method at 40% p.a. on nominal capex
# WDV at end of year = nominal_capex * (1 - 0.40)^(years_of_depreciation)
# First year gets full depreciation since capex is on first day

def tax_closing_balance(target_year):
    """Calculate closing taxation asset balance at end of target_year."""
    total = 0.0
    for vy in years:
        if vy > target_year:
            continue
        nc = nominal_capex[vy]
        years_depr = target_year - vy + 1  # full year in first year
        wdv = nc * (1 - tax_dep_rate) ** years_depr
        total += wdv
    return total

print("Taxation Asset Schedule - Closing Balances:")
tax_balances = {}
for y in years:
    bal = tax_closing_balance(y)
    tax_balances[y] = bal
    print(f"  {y}: {bal:.1f}m")

In [ ]:
# ============================================================
# QUESTION 3: Closing balance for taxation asset schedule 2019
# ============================================================
# Options: a) 101.6m  b) 62.9m  c) 91.8m  d) 68.2

tax_close_2019 = tax_closing_balance(2019)
print(f"Taxation closing balance 2019: {tax_close_2019:.1f}m")

options_q3 = {'a': 101.6, 'b': 62.9, 'c': 91.8, 'd': 68.2}
q3_answer = min(options_q3, key=lambda k: abs(options_q3[k] - tax_close_2019))
print(f"Closest match: {q3_answer}) {options_q3[q3_answer]}m")

In [ ]:
# ============================================================
# QUESTION 4: Impact of change in taxation rate on opening
#              deferred tax balance for 2016
# ============================================================
# Options: a) 2.2m  b) 12.6m  c) 7.5m  d) 9.7
#
# Opening balance for 2016 = Closing balance for 2015
# The deferred tax balance at end of 2015 was calculated at 30%
# From 2016 the rate changes to 28%
# Impact = (Acct_WDV - Tax_WDV) * (new_rate - old_rate)
#        = (Acct_WDV - Tax_WDV) * (0.28 - 0.30)
#        = (Acct_WDV - Tax_WDV) * (-0.02)

acct_close_2015 = acct_closing_balance(2015)
tax_close_2015 = tax_closing_balance(2015)
diff_2015 = acct_close_2015 - tax_close_2015

print(f"Accounting closing balance 2015: {acct_close_2015:.4f}m")
print(f"Taxation closing balance 2015: {tax_close_2015:.4f}m")
print(f"Difference (Acct - Tax): {diff_2015:.4f}m")

# Deferred tax at old rate (30%)
dt_old = diff_2015 * 0.30
# Deferred tax at new rate (28%)
dt_new = diff_2015 * 0.28
# Impact of rate change
impact = abs(dt_new - dt_old)

print(f"DT at 30%: {dt_old:.4f}m")
print(f"DT at 28%: {dt_new:.4f}m")
print(f"Impact of rate change: {impact:.1f}m")

options_q4 = {'a': 2.2, 'b': 12.6, 'c': 7.5, 'd': 9.7}
q4_answer = min(options_q4, key=lambda k: abs(options_q4[k] - impact))
print(f"Closest match: {q4_answer}) {options_q4[q4_answer]}m")

In [ ]:
# ============================================================
# QUESTION 5: Deferred tax balance as at 2028
# ============================================================
# Options: a) 79.0m Asset  b) 79.0m Liability  c) 80.2m Asset  d) 80.2m Liability
#
# Deferred tax = (Accounting WDV - Tax WDV) * tax_rate
# If positive (Acct > Tax) => Liability
# If negative (Acct < Tax) => Asset
# Tax rate from 2016 onwards is 28%

acct_close_2028 = acct_closing_balance(2028)
tax_close_2028 = tax_closing_balance(2028)
diff_2028 = acct_close_2028 - tax_close_2028
dt_2028 = diff_2028 * 0.28

print(f"Accounting closing balance 2028: {acct_close_2028:.4f}m")
print(f"Taxation closing balance 2028: {tax_close_2028:.4f}m")
print(f"Difference (Acct - Tax): {diff_2028:.4f}m")
print(f"Deferred tax at 28%: {dt_2028:.4f}m")

dt_type = 'Liability' if dt_2028 > 0 else 'Asset'
print(f"Type: {abs(dt_2028):.1f}m {dt_type}")

options_q5 = {
    'a': (79.0, 'Asset'),
    'b': (79.0, 'Liability'),
    'c': (80.2, 'Asset'),
    'd': (80.2, 'Liability')
}
q5_answer = min(options_q5, key=lambda k: abs(options_q5[k][0] - abs(dt_2028)) + (0 if options_q5[k][1] == dt_type else 1000))
print(f"Closest match: {q5_answer}) {options_q5[q5_answer][0]}m {options_q5[q5_answer][1]}")

In [ ]:
# ============================================================
# FINAL ANSWERS
# ============================================================

answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
}

print("Final Answers:")
for k, v in answers.items():
    print(f"  {k}: {v}")